# 02 · 读取训练产物

**路径：入门 Intro。** 运行一个最小训练并读取 `metrics.json`、`history.csv` 与预测产物。

本页运行的是固定随机种子、CPU、小步数 **smoke**，只验证完整训练与结果链路，不是精度结论。
浏览器中的方法配置器仍是并行入口：它下载可训练 `.py`，不会在浏览器训练或上传数据。
notebook 源文件位于 `examples/tutorials/`，可下载后在仓库根目录重新执行。


In [1]:
from pathlib import Path
import json
import sys

import openpinn as op

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("OpenPINN", op.__version__, "| repo", ROOT.name)


OpenPINN 0.2.0 | repo OpenPINN-upstream


In [2]:
result = op.solve(
    "poisson", method="baseline", steps=2, seed=1, device="cpu",
    model={"hidden_layers": [8, 8]},
    sampling={"interior": 12, "boundary": 4},
    overrides={"evaluation": {"points": 17}},
    output_dir=Path("/tmp/openpinn-m17-intro-artifacts"), plot=False,
)
print("training complete:", result.output_dir.name)


step=1 total=5.562943e+01
step=2 total=4.968861e+01
training complete: openpinn-m17-intro-artifacts


In [3]:
metrics = json.loads((result.output_dir / "metrics.json").read_text(encoding="utf-8"))
artifacts = sorted(path.name for path in result.output_dir.iterdir())
print(json.dumps({
    "final_total_loss": metrics["final_total_loss"],
    "history_rows": len((result.output_dir / "history.csv").read_text().splitlines()) - 1,
    "has_predictions": "predictions.npz" in artifacts,
    "has_checkpoint": "checkpoint.pt" in artifacts,
}, ensure_ascii=False, indent=2))


{
  "final_total_loss": 49.68861389160156,
  "history_rows": 2,
  "has_predictions": true,
  "has_checkpoint": true
}
